# Notebook 8 — Export GPX propre

## Objectif

Convertir les itinéraires des notebooks 4 et 6 en fichiers GPX 1.1 valides,
exploitables sur Garmin, Wahoo, Komoot, etc.

## Fonctionnalités

- Conversion path → GPX avec waypoints
- Lissage des zigzags (élimine retours arrière < 30m)
- Densification optionnelle (1 point tous les 25m pour Garmin)
- Métadonnées GPX : nom, description, distance, D+
- Reproject vers WGS84 (standard GPX)


## 1. Setup

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import time
from datetime import datetime
from pathlib import Path
from math import radians, sin, cos, atan2, sqrt
from sqlalchemy import create_engine, text
from shapely import wkt
import warnings
warnings.filterwarnings("ignore")

DB_CONFIG = {"user": "postgres", "password": "4421",
             "host": "localhost", "port": 5432, "database": "velo_club"}
url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
       f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(url, pool_pre_ping=True)

OUT_DIR = Path("data/gpx_exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OK")

OK


## 2. Utilitaires

In [1]:
def haversine_m(p1, p2):
    """Distance haversine en mètres."""
    R = 6371000
    phi1, phi2 = radians(p1[0]), radians(p2[0])
    dphi = radians(p2[0] - p1[0])
    dlam = radians(p2[1] - p1[1])
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlam/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def smooth_path(coords, min_step_m=30):
    """Lisse les zigzags : élimine les points trop proches du précédent."""
    if len(coords) < 3:
        return coords
    smoothed = [coords[0]]
    for pt in coords[1:-1]:
        if haversine_m(smoothed[-1], pt) >= min_step_m:
            smoothed.append(pt)
    smoothed.append(coords[-1])
    return smoothed


def densify_path(coords, target_step_m=25):
    """Insère des points intermédiaires pour avoir un point tous les ~25m."""
    if len(coords) < 2:
        return coords
    dense = [coords[0]]
    for i in range(1, len(coords)):
        p1 = dense[-1]
        p2 = coords[i]
        d = haversine_m(p1, p2)
        if d > target_step_m:
            n_inserts = int(d // target_step_m)
            for j in range(1, n_inserts + 1):
                t = j / (n_inserts + 1)
                interp = (p1[0] + t * (p2[0] - p1[0]),
                          p1[1] + t * (p2[1] - p1[1]))
                dense.append(interp)
        dense.append(p2)
    return dense


def get_elevations(coords):
    """Récupère l'altitude pour chaque point depuis SRTM (osm_edges sample).
    Approximation : on prend l'altitude des arêtes osm_edges les plus proches.
    """
    if not coords:
        return []
    # Pour V1, on retourne 0 partout. À améliorer avec SRTM si besoin.
    return [0.0] * len(coords)


def compute_route_stats(coords):
    """Calcule distance et D+ approximatif."""
    if len(coords) < 2:
        return 0, 0
    total_dist = sum(haversine_m(coords[i], coords[i+1]) for i in range(len(coords)-1))
    return total_dist, 0  # D+ à calculer plus précisément si besoin

print("Utils OK")

Utils OK


## 3. Conversion en GPX

In [3]:
def export_gpx(coords, name="Itineraire", description="",
               filepath=None, smooth=True, densify=True,
               elevations=None):
    """
    Convertit une liste de (lat, lon) en fichier GPX 1.1.
    
    Args:
        coords: [(lat, lon), ...]
        name: nom de la trace
        description: description GPX
        filepath: chemin de sortie (.gpx). Si None, retourne le XML string
        smooth: appliquer le lissage anti-zigzag
        densify: densifier pour Garmin (1pt/25m)
        elevations: liste d'altitudes en m (optionnel)
    
    Returns:
        Path du fichier créé (ou XML string si filepath=None)
    """
    if smooth:
        coords = smooth_path(coords, min_step_m=30)
    if densify:
        coords = densify_path(coords, target_step_m=25)
    
    if elevations is None:
        elevations = get_elevations(coords)
    if len(elevations) != len(coords):
        elevations = [0.0] * len(coords)
    
    total_dist, d_plus = compute_route_stats(coords)
    timestamp = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    
    # XML GPX 1.1
    xml = ['<?xml version="1.0" encoding="UTF-8"?>']
    xml.append('<gpx version="1.1" creator="VeloClubIDF" '
               'xmlns="http://www.topografix.com/GPX/1/1" '
               'xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" '
               'xsi:schemaLocation="http://www.topografix.com/GPX/1/1 '
               'http://www.topografix.com/GPX/1/1/gpx.xsd">')
    xml.append('  <metadata>')
    xml.append(f'    <name>{name}</name>')
    xml.append(f'    <desc>{description} - {total_dist/1000:.1f} km</desc>')
    xml.append(f'    <time>{timestamp}</time>')
    xml.append('  </metadata>')
    xml.append('  <trk>')
    xml.append(f'    <name>{name}</name>')
    xml.append('    <trkseg>')
    
    for (lat, lon), ele in zip(coords, elevations):
        xml.append(f'      <trkpt lat="{lat:.7f}" lon="{lon:.7f}">')
        xml.append(f'        <ele>{ele:.1f}</ele>')
        xml.append('      </trkpt>')
    
    xml.append('    </trkseg>')
    xml.append('  </trk>')
    xml.append('</gpx>')
    
    content = "\n".join(xml)
    
    if filepath:
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"✓ GPX écrit : {filepath} ({len(coords)} points, {total_dist/1000:.1f} km)")
        return Path(filepath)
    else:
        return content

print("export_gpx OK")

export_gpx OK


## 4. Test sur une trace simple

In [4]:
# Test simple : 3 points
test_coords = [(48.8566, 2.3522), (48.85, 2.35), (48.84, 2.36)]
gpx = export_gpx(test_coords, name="Test", 
                 filepath=OUT_DIR / "test.gpx", 
                 smooth=False, densify=False)
print("\nContenu :")
with open(gpx) as f:
    print(f.read()[:500])

✓ GPX écrit : data\gpx_exports\test.gpx (3 points, 2.1 km)

Contenu :
<?xml version="1.0" encoding="UTF-8"?>
<gpx version="1.1" creator="VeloClubIDF" xmlns="http://www.topografix.com/GPX/1/1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.topografix.com/GPX/1/1 http://www.topografix.com/GPX/1/1/gpx.xsd">
  <metadata>
    <name>Test</name>
    <desc> - 2.1 km</desc>
    <time>2026-05-15T17:54:21Z</time>
  </metadata>
  <trk>
    <name>Test</name>
    <trkseg>
      <trkpt lat="48.8566000" lon="2.3522000">
        <ele>0.0</ele>


## 5. Export depuis Notebook 4

Suppose qu'on a un objet `route_result` issu du Notebook 4 ou 6.

In [5]:
# Exemple d'utilisation après Notebook 4
def export_route_result(result, name, output_dir=OUT_DIR):
    """Export un résultat de route() en GPX."""
    coords = result["coords"]
    distance_km = result["total_length_m"] / 1000
    score = result.get("mean_score", 0)
    profile = result.get("profile", "?")
    alpha = result.get("alpha", "?")
    
    filename = f"{name}_{profile}_a{alpha}.gpx"
    desc = f"{distance_km:.1f}km, score {score:.2f}, profil {profile}, α={alpha}"
    
    return export_gpx(
        coords,
        name=name,
        description=desc,
        filepath=output_dir / filename,
        smooth=True,
        densify=True,
    )

# Exemple
# r = route(G, POINTS["pantin"], POINTS["chevreuse"], profile="club_road", alpha=1.0)
# export_route_result(r, "pantin_chevreuse")
print("export_route_result OK - utilise après avoir calculé un itinéraire")

export_route_result OK - utilise après avoir calculé un itinéraire


## 6. Bulk export

Export plusieurs itinéraires d'un coup pour ton mémoire / soutenance.

In [6]:
# À utiliser depuis le notebook 4 ou 6 après avoir construit G
"""
PAIRS = [
    ("pantin", "chevreuse"),
    ("pantin", "fontainebleau"),
    ("pantin", "rambouillet"),
    ("pantin", "cergy"),
]

for src, dst in PAIRS:
    for profile in ["club_road", "solo_casual"]:
        for alpha in [1.0, 1.5]:
            r = route(G, POINTS[src], POINTS[dst], profile=profile, alpha=alpha)
            if r:
                export_route_result(r, f"{src}_{dst}_a{alpha}")
"""
print("Décommente ce bloc dans le notebook 4 pour bulk export")

Décommente ce bloc dans le notebook 4 pour bulk export


## Notes

- Le lissage à 30m évite les zigzags entre arêtes parallèles
- La densification à 25m est nécessaire pour Garmin Edge (sinon il interpole en ligne droite et tu rates les virages)
- Pour avoir les vraies altitudes, il faudrait recroiser avec le SRTM (à faire en V2)
- Le GPX produit est compatible Komoot, Strava, Garmin Connect, Wahoo
